In [4]:
import torch
import triton
import triton.language as tl
import torch.nn.functional as F


@triton.jit
def _chunk_gated_delta_rule_prefill_kernel(
    Q, K, V, BETA, G,
    INIT_STATE,
    OUT,
    FINAL_STATE,

    B: tl.constexpr,
    T: tl.constexpr,
    H: tl.constexpr,
    DK: tl.constexpr,
    DV: tl.constexpr,
    N_CHUNKS: tl.constexpr,

    SCALE: tl.constexpr,
    HAS_INIT: tl.constexpr,
    SAVE_FINAL_STATE: tl.constexpr,

    CHUNK: tl.constexpr,
    BLOCK_DK: tl.constexpr,
    BLOCK_DV: tl.constexpr,
):
    pid_bh = tl.program_id(0)
    pid_vb = tl.program_id(1)

    b = pid_bh // H
    h = pid_bh - b * H

    offs_m = tl.arange(0, CHUNK)
    offs_n = tl.arange(0, CHUNK)
    offs_dk = tl.arange(0, BLOCK_DK)
    offs_dv = pid_vb * BLOCK_DV + tl.arange(0, BLOCK_DV)

    dk_mask = offs_dk < DK
    dv_mask = offs_dv < DV

    # state: [DK, BLOCK_DV]
    if HAS_INIT:
        init_ptrs = (
            INIT_STATE
            + b * H * DK * DV
            + h * DK * DV
            + offs_dk[:, None] * DV
            + offs_dv[None, :]
        )
        state = tl.load(
            init_ptrs,
            mask=dk_mask[:, None] & dv_mask[None, :],
            other=0.0,
        ).to(tl.float32)
    else:
        state = tl.zeros((BLOCK_DK, BLOCK_DV), dtype=tl.float32)

    row_id = offs_m[:, None]
    col_id = offs_n[None, :]

    for cid in tl.range(0, N_CHUNKS):
        t_offsets = cid * CHUNK + offs_m
        t_mask = t_offsets < T

        # ------------------------------------------------------------
        # load q, k: [CHUNK, DK]
        # load v:    [CHUNK, BLOCK_DV]
        # load beta, g: [CHUNK]
        # ------------------------------------------------------------
        q_ptrs = (
            Q
            + b * T * H * DK
            + t_offsets[:, None] * H * DK
            + h * DK
            + offs_dk[None, :]
        )
        k_ptrs = (
            K
            + b * T * H * DK
            + t_offsets[:, None] * H * DK
            + h * DK
            + offs_dk[None, :]
        )
        v_ptrs = (
            V
            + b * T * H * DV
            + t_offsets[:, None] * H * DV
            + h * DV
            + offs_dv[None, :]
        )
        beta_ptrs = BETA + b * T * H + t_offsets * H + h
        g_ptrs = G + b * T * H + t_offsets * H + h

        q = tl.load(
            q_ptrs,
            mask=t_mask[:, None] & dk_mask[None, :],
            other=0.0,
        ).to(tl.float32) * SCALE

        k = tl.load(
            k_ptrs,
            mask=t_mask[:, None] & dk_mask[None, :],
            other=0.0,
        ).to(tl.float32)

        v = tl.load(
            v_ptrs,
            mask=t_mask[:, None] & dv_mask[None, :],
            other=0.0,
        ).to(tl.float32)

        beta = tl.load(beta_ptrs, mask=t_mask, other=0.0).to(tl.float32)
        g_raw = tl.load(g_ptrs, mask=t_mask, other=0.0).to(tl.float32)

        # chunk-local cumulative g
        g_cum = tl.cumsum(g_raw, 0)

        beta_col = beta[:, None]
        k_beta = k * beta_col
        v_beta = v * beta_col

        # ------------------------------------------------------------
        # decay_mask[i, j] = exp(g[i] - g[j]) if i >= j else 0
        # ------------------------------------------------------------
        g_diff = g_cum[:, None] - g_cum[None, :]
        lower_eq = row_id >= col_id
        lower_strict = row_id > col_id

        decay = tl.exp(g_diff)
        decay = tl.where(lower_eq, decay, 0.0)

        # ------------------------------------------------------------
        # attn = -((k_beta @ k.T) * decay).masked_fill(triu(diagonal=0), 0)
        # i.e. only lower-strict part is kept.
        # ------------------------------------------------------------
        kk = tl.dot(k_beta, tl.trans(k))
        attn = -kk * decay
        attn = tl.where(lower_strict, attn, 0.0)

        # ------------------------------------------------------------
        # for i in range(1, CHUNK):
        #     row = attn[i, :i]
        #     sub = attn[:i, :i]
        #     attn[i, :i] = row + (row.unsqueeze(-1) * sub).sum(-2)
        # ------------------------------------------------------------
        # 这里只能用左上角前缀块，不能把未来行也卷进 update 里。
        for ii in tl.range(1, CHUNK):
            is_row = offs_m == ii
            prefix_mask = (row_id < ii) & (col_id < ii)

            row = tl.sum(
                tl.where(is_row[:, None], attn, 0.0),
                axis=0,
            )

            update = tl.sum(row[:, None] * tl.where(prefix_mask, attn, 0.0), axis=0)
            new_row = row + update

            attn = tl.where(
                (row_id == ii) & (col_id < ii),
                new_row[None, :],
                attn,
            )

        eye = row_id == col_id
        attn_eye = attn + tl.where(eye, 1.0, 0.0)

        # value = attn @ v_beta
        v_local = tl.dot(attn_eye, v_beta)

        # k_cumdecay = attn @ (k_beta * exp(g))
        k_decay = k_beta * tl.exp(g_cum)[:, None]
        k_cumdecay = tl.dot(attn_eye, k_decay)

        # ------------------------------------------------------------
        # recurrent part
        # v_prime = k_cumdecay @ state
        # v_new = value - v_prime
        # ------------------------------------------------------------
        v_prime = tl.dot(k_cumdecay, state)
        v_new = v_local - v_prime

        # attn = (q @ k.T * decay).masked_fill(triu(diagonal=1), 0)
        qk = tl.dot(q, tl.trans(k))
        causal_attn = qk * decay
        causal_attn = tl.where(lower_eq, causal_attn, 0.0)

        # attn_inter = (q * exp(g)) @ state
        q_decay = q * tl.exp(g_cum)[:, None]
        attn_inter = tl.dot(q_decay, state)

        out = attn_inter + tl.dot(causal_attn, v_new)

        # store output: [B, T, H, DV]
        out_ptrs = (
            OUT
            + b * T * H * DV
            + t_offsets[:, None] * H * DV
            + h * DV
            + offs_dv[None, :]
        )
        tl.store(
            out_ptrs,
            out,
            mask=t_mask[:, None] & dv_mask[None, :],
        )

        # ------------------------------------------------------------
        # last_recurrent_state =
        #     state * exp(g_last)
        #     + (k_i * exp(g_last - g_i)).T @ v_new
        # ------------------------------------------------------------
        g_last = tl.sum(
            tl.where(offs_m == CHUNK - 1, g_cum, 0.0),
            axis=0,
        )

        k_state = k * tl.exp(g_last - g_cum)[:, None]
        state = state * tl.exp(g_last) + tl.dot(tl.trans(k_state), v_new)

    if SAVE_FINAL_STATE:
        final_ptrs = (
            FINAL_STATE
            + b * H * DK * DV
            + h * DK * DV
            + offs_dk[:, None] * DV
            + offs_dv[None, :]
        )
        tl.store(
            final_ptrs,
            state,
            mask=dk_mask[:, None] & dv_mask[None, :],
        )
def triton_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=16,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
    block_dv=16,
):
    """
    query: [B, T, H, Dk]
    key:   [B, T, H, Dk]
    value: [B, T, H, Dv]
    g:     [B, T, H]
    beta:  [B, T, H]
    """

    assert query.ndim == 4
    assert key.ndim == 4
    assert value.ndim == 4
    assert beta.ndim == 3
    assert g.ndim == 3

    B, T, H, DK = query.shape
    _, _, _, DV = value.shape

    assert key.shape == query.shape
    assert value.shape[:3] == query.shape[:3]
    assert beta.shape == (B, T, H)
    assert g.shape == (B, T, H)

    initial_dtype = query.dtype

    if use_qk_l2norm_in_kernel:
        # 这里先用 PyTorch 做，后面可以单独融合到 kernel 里。
        query = F.normalize(query.float(), p=2, dim=-1, eps=1e-6).to(initial_dtype)
        key = F.normalize(key.float(), p=2, dim=-1, eps=1e-6).to(initial_dtype)

    query = query.contiguous()
    key = key.contiguous()
    value = value.contiguous()
    beta = beta.contiguous()
    g = g.contiguous()

    out = torch.empty_like(value)

    if initial_state is not None:
        assert initial_state.shape == (B, H, DK, DV)
        initial_state = initial_state.contiguous()
        has_init = True
    else:
        initial_state = torch.empty((0,), device=query.device, dtype=torch.float32)
        has_init = False

    if output_final_state:
        final_state = torch.empty(
            (B, H, DK, DV),
            device=query.device,
            dtype=torch.float32,
        )
    else:
        final_state = torch.empty((0,), device=query.device, dtype=torch.float32)

    n_chunks = triton.cdiv(T, chunk_size)
    block_dk = triton.next_power_of_2(DK)

    assert chunk_size in (16, 32), "先用较小 chunk_size 做 sanity check，稳定后再放大。"
    assert block_dk <= 256, "DK 太大时单 kernel register 压力会很高，建议拆 DK。"
    assert block_dv in (16, 32), "当前 tl.dot 路径要求 block_dv >= 16。"

    scale = 1.0 / (DK ** 0.5)

    grid = (B * H, triton.cdiv(DV, block_dv))

    num_warps = 2 if chunk_size <= 16 and block_dv <= 16 else 4

    _chunk_gated_delta_rule_prefill_kernel[grid](
        query,
        key,
        value,
        beta,
        g,
        initial_state,
        out,
        final_state,

        B,
        T,
        H,
        DK,
        DV,
        n_chunks,

        scale,
        has_init,
        output_final_state,

        CHUNK=chunk_size,
        BLOCK_DK=block_dk,
        BLOCK_DV=block_dv,

        num_warps=num_warps,
        num_stages=3,
    )

    if not output_final_state:
        final_state = None

    return out, final_state

In [5]:

def l2norm(x: torch.FloatTensor, dim: int = -1, eps: float = 1e-6):
    """This function is intended to align with the l2norm implementation in the FLA library."""
    inv_norm = torch.rsqrt((x * x).sum(dim=dim, keepdim=True) + eps)
    return x * inv_norm
def torch_chunk_gated_delta_rule(
    query,
    key,
    value,
    g,
    beta,
    chunk_size=16,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=False,
):
    initial_dtype = query.dtype
    if use_qk_l2norm_in_kernel:
        query = l2norm(query, dim=-1, eps=1e-6)
        key = l2norm(key, dim=-1, eps=1e-6)
    query, key, value, beta, g = [
        x.transpose(1, 2).contiguous().to(torch.float32) for x in (query, key, value, beta, g)
    ]

    batch_size, num_heads, sequence_length, k_head_dim = key.shape
    v_head_dim = value.shape[-1]
    pad_size = (chunk_size - sequence_length % chunk_size) % chunk_size
    query = F.pad(query, (0, 0, 0, pad_size))
    key = F.pad(key, (0, 0, 0, pad_size))
    value = F.pad(value, (0, 0, 0, pad_size))
    beta = F.pad(beta, (0, pad_size))
    g = F.pad(g, (0, pad_size))
    total_sequence_length = sequence_length + pad_size
    scale = 1 / (query.shape[-1] ** 0.5)
    query = query * scale

    v_beta = value * beta.unsqueeze(-1)
    k_beta = key * beta.unsqueeze(-1)
    # reshape to chunks
    query, key, value, k_beta, v_beta = [
        x.reshape(x.shape[0], x.shape[1], -1, chunk_size, x.shape[-1]) for x in (query, key, value, k_beta, v_beta)
    ]
    g = g.reshape(g.shape[0], g.shape[1], -1, chunk_size)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=0)

    # chunk decay
    g = g.cumsum(dim=-1)
    decay_mask = ((g.unsqueeze(-1) - g.unsqueeze(-2)).tril().exp().float()).tril()
    attn = -((k_beta @ key.transpose(-1, -2)) * decay_mask).masked_fill(mask, 0)
    for i in range(1, chunk_size):
        row = attn[..., i, :i].clone()
        sub = attn[..., :i, :i].clone()
        attn[..., i, :i] = row + (row.unsqueeze(-1) * sub).sum(-2)
    attn = attn + torch.eye(chunk_size, dtype=attn.dtype, device=attn.device)
    value = attn @ v_beta
    k_cumdecay = attn @ (k_beta * g.exp().unsqueeze(-1))
    last_recurrent_state = (
        torch.zeros(batch_size, num_heads, k_head_dim, v_head_dim).to(value)
        if initial_state is None
        else initial_state.to(value)
    )
    core_attn_out = torch.zeros_like(value)
    mask = torch.triu(torch.ones(chunk_size, chunk_size, dtype=torch.bool, device=query.device), diagonal=1)

    # for each chunk
    for i in range(0, total_sequence_length // chunk_size):
        q_i, k_i, v_i = query[:, :, i], key[:, :, i], value[:, :, i]
        attn = q_i @ k_i.transpose(-1, -2) * decay_mask[:, :, i]
        v_prime = (k_cumdecay[:, :, i]) @ last_recurrent_state
        v_new = v_i - v_prime
        attn_inter = (q_i * g[:, :, i, :, None].exp()) @ last_recurrent_state
        core_attn_out[:, :, i] = attn_inter + attn @ v_new
        last_recurrent_state = (
            last_recurrent_state * g[:, :, i, -1, None, None].exp()
            + (k_i * (g[:, :, i, -1, None] - g[:, :, i]).exp()[..., None]).transpose(-1, -2) @ v_new
        )

    if not output_final_state:
        last_recurrent_state = None
    core_attn_out = core_attn_out.reshape(core_attn_out.shape[0], core_attn_out.shape[1], -1, core_attn_out.shape[-1])
    core_attn_out = core_attn_out[:, :, :sequence_length]
    core_attn_out = core_attn_out.transpose(1, 2).contiguous().to(initial_dtype)
    return core_attn_out, last_recurrent_state


In [6]:
torch.manual_seed(0)

B, T, H, DK, DV = 1, 32, 2, 16, 16
chunk_size = 16
device = "cuda"
dtype = torch.float32

q = torch.randn(B, T, H, DK, device=device, dtype=dtype) * 0.1
k = torch.randn(B, T, H, DK, device=device, dtype=dtype) * 0.1
v = torch.randn(B, T, H, DV, device=device, dtype=dtype) * 0.1

# 这里显式取非正数，避免 exp 在 sanity check 里把数值放大得太狠。
g = -torch.rand(B, T, H, device=device, dtype=dtype) * 0.001
beta = torch.sigmoid(torch.randn(B, T, H, device=device, dtype=dtype))

out_ref, state_ref = torch_chunk_gated_delta_rule(
    q,
    k,
    v,
    g,
    beta,
    chunk_size=chunk_size,
    initial_state=None,
    output_final_state=True,
)

out_tri, state_tri = triton_chunk_gated_delta_rule(
    q,
    k,
    v,
    g,
    beta,
    chunk_size=chunk_size,
    initial_state=None,
    output_final_state=True,
    block_dv=16,
)
torch.cuda.synchronize()

print("shapes:", out_ref.shape, out_tri.shape, state_ref.shape, state_tri.shape)

print("out max abs error:", (out_ref.float() - out_tri.float()).abs().max())
print("out mean abs error:", (out_ref.float() - out_tri.float()).abs().mean())
print("state max abs error:", (state_ref.float() - state_tri.float()).abs().max())
print("state mean abs error:", (state_ref.float() - state_tri.float()).abs().mean())

shapes: torch.Size([1, 32, 2, 16]) torch.Size([1, 32, 2, 16]) torch.Size([1, 2, 16, 16]) torch.Size([1, 2, 16, 16])
out max abs error: tensor(1.9592e-05, device='cuda:0')
out mean abs error: tensor(2.7247e-06, device='cuda:0')
state max abs error: tensor(9.3587e-05, device='cuda:0')
state mean abs error: tensor(2.3547e-05, device='cuda:0')
